# Counter-Strike Winner Predictor — Training, Evaluation & Validation

End-to-end notebook: load data → feature engineering → train/val/test → full evaluation → SHAP → pre-match prediction demo.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.module.model.data import load_raw_data, preprocess
from src.module.model.features import build_features, get_feature_matrix, build_team_stats_lookup, FEATURE_COLS
from src.module.model.model import CSWinnerModel
from src.module.model.evaluate import ModelEvaluator
from src.module.model.explain import ModelExplainer
from utils.data_processing import split_data
from utils.logger import get_logger

logger = get_logger('notebook')
sns.set_theme(style='whitegrid')
%matplotlib inline
print(f'Feature columns: {len(FEATURE_COLS)}')

## 1. Load & Preprocess

In [ ]:
raw = load_raw_data()
df  = preprocess(raw)
print(f'Dataset: {df.shape[0]} rows × {df.shape[1]} cols')
df.head(3)

## 2. Feature Engineering

In [ ]:
df = build_features(df)
print(f'After feature engineering: {df.shape[0]} rows × {df.shape[1]} cols')
print(f'Pre-match FEATURE_COLS ({len(FEATURE_COLS)}):', FEATURE_COLS)

## 3. Target Distribution

In [ ]:
df['winner'].value_counts().rename({0: 'team2 wins', 1: 'team1 wins'}).plot.bar(
    color=['#e74c3c', '#2ecc71'], figsize=(5, 3)
)
plt.title('Match outcome distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()

## 4. Feature Correlation with Winner

In [ ]:
X, y = get_feature_matrix(df)
corr = X.corrwith(y).sort_values()
corr.plot.barh(figsize=(8, max(6, len(corr) // 3)), title='Feature correlation with winner')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()

## 5. Feature Distribution (top correlated)

In [ ]:
top_feats = corr.abs().sort_values(ascending=False).head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, feat in zip(axes.flat, top_feats):
    for label, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
        X[y == label][feat].dropna().plot.kde(ax=ax, color=color, label=f'winner={label}')
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('Feature distributions by match outcome', y=1.01)
plt.tight_layout()

## 6. Train / Validation / Test Split

In [ ]:
from conf.settings import TEST_SIZE, VAL_SIZE, RANDOM_STATE

X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y, test_size=TEST_SIZE, val_size=VAL_SIZE, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')
print(f'Class balance — Train: {y_train.mean():.3f}  Val: {y_val.mean():.3f}  Test: {y_test.mean():.3f}')

## 7. Cross-Validation

In [ ]:
model = CSWinnerModel()
cv_results = model.cross_validate(X_train, y_train)
print(f"CV AUC: {cv_results['mean_auc']:.4f} ± {cv_results['std_auc']:.4f}")

## 8. Train Final Model

In [ ]:
model = CSWinnerModel()
model.fit(X_train, y_train, X_val=X_val, y_val=y_val)
model.save()
print('Model saved.')

## 9. Full Evaluation on Test Set

In [ ]:
evaluator = ModelEvaluator(model)
metrics = evaluator.compute_metrics(X_test, y_test)
print('\nTest set metrics:')
for k, v in metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 10. Confusion Matrix

In [ ]:
evaluator.plot_confusion_matrix(X_test, y_test, filename='confusion_matrix.png')

## 11. ROC Curve

In [ ]:
evaluator.plot_roc_curve(X_test, y_test, filename='roc_curve.png')

## 12. Validation Set Metrics

In [ ]:
val_metrics = evaluator.compute_metrics(X_val, y_val)
print('Validation set metrics:')
for k, v in val_metrics.items():
    print(f'  {k:12s}: {v:.4f}')

## 13. SHAP — Global Feature Importance

In [ ]:
evaluator.run_shap(df, filename='shap_summary.png')

## 14. SHAP — Explain One Match (Local Waterfall)

In [ ]:
# Use the row with the highest predicted probability for team1 as an interesting example
X_sample = X_test.copy()
probas = model.pipeline.predict_proba(X_sample)[:, 1]
most_confident_idx = probas.argmax()

sample_row = df.iloc[X_test.index[most_confident_idx]]
explainer = ModelExplainer(model=model)
result = explainer.explain_match(sample_row.to_dict(), save=True, filename='shap_waterfall.png')
print('Predicted winner:', result['predicted_winner'])
print(f"Prob team1: {result['prob_team1_wins']:.4f}  Prob team2: {result['prob_team2_wins']:.4f}")
print('\nTop reasons:')
for r in result['top_reasons']:
    print(f"  {r['feature']:35s} SHAP={r['shap_value']:+.4f}  val={r['feature_value']:.4f}")

## 15. Pre-Match Prediction API Demo

In [ ]:
# Build the historical lookup (do this once per session)
lookup = build_team_stats_lookup(df)
teams = sorted(lookup.keys())
print(f'Teams in lookup: {len(teams)}')
print('Sample teams:', teams[:10])

In [ ]:
# Predict a match — change team names to any teams in the dataset
team1 = teams[0]
team2 = teams[1]

result = model.predict_match(
    team1=team1,
    team2=team2,
    match_format='bo3',
    stage='Grand Final',
    team_stats=lookup,
)

print(f"\n{'='*40}")
print(f"  {result['team1']} vs {result['team2']}")
print(f"  Predicted winner: {result['predicted_winner']}")
print(f"  P(team1 wins): {result['prob_team1_wins']:.1%}")
print(f"  P(team2 wins): {result['prob_team2_wins']:.1%}")
print(f"  ELO diff: {result['elo_diff']}")
print(f"{'='*40}")

## 16. ELO Rating Leaderboard

In [ ]:
# Show top teams by ELO
elo_rows = [
    {'team': team, 'elo': stats.get('elo', 1000), 'overall_wr': stats.get('overall_wr', float('nan'))}
    for team, stats in lookup.items()
]
elo_df = pd.DataFrame(elo_rows).sort_values('elo', ascending=False).reset_index(drop=True)
print('Top 20 teams by ELO:')
elo_df.head(20).style.format({'elo': '{:.1f}', 'overall_wr': '{:.3f}'})

## 17. Compare Train vs Val vs Test Metrics

In [ ]:
train_metrics = evaluator.compute_metrics(X_train, y_train)

summary = pd.DataFrame({
    'Train': train_metrics,
    'Val':   val_metrics,
    'Test':  metrics,
}).T

print('\nTrain / Val / Test comparison:')
summary.style.format('{:.4f}').background_gradient(axis=0)